# Imports

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from time import perf_counter

from sympy.physics.units import temperature

if Path("/kaggle/working").is_dir():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers==4.57.6", "accelerate==1.12.0", "bitsandbytes==0.49.2"])

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

# Reader RuadaptQwen3-8B-Hybrid

In [ ]:
MODEL_ID = "RefalMachine/RuadaptQwen3-8B-Hybrid"
REVISION = "b827af5797f263245939190dc1e4fd54565b3405"
if not torch.cuda.is_available():
    raise RuntimeError("Для этого эксперимента нужен CUDA GPU.")

torch.cuda.reset_peak_memory_stats(0)
started = perf_counter()
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, quantization_config=quantization,
    device_map={"": 0}, dtype=torch.float16,
)
reader = pipeline("text-generation", model=model, tokenizer=tokenizer)
torch.cuda.synchronize(0)
load_seconds = perf_counter() - started

In [ ]:
refusal = "В предоставленном контексте нет ответа."
title = "Договор Альфа"
context = f"[{title}] Покупатель оплачивает заказ в течение 17 рабочих дней."
cases = [
    ("grounded", "В какой срок покупатель оплачивает заказ?"),
    ("missing", "Какой штраф предусмотрен за просрочку?"),
]
results = []
for name, question in cases:
    messages = [
        {"role": "system", "content": (
            "Отвечай только по контексту. Приводи название источника в квадратных скобках. "
            f"Если данных для ответа нет, ответь ровно так: {refusal}"
        )},
        {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"},
    ]
    prompt = (tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    ))
    started = perf_counter()
    output = reader(prompt, do_sample=False, return_full_text=False,
                    max_new_tokens=160, add_special_tokens=False)
    torch.cuda.synchronize(0)
    elapsed = perf_counter() - started
    answer = output[0]["generated_text"].strip()
    passed = ("17 рабочих дней" in answer.lower() and f"[{title}]" in answer
              if name == "grounded" else answer == refusal)
    results.append(dict(case=name, answer=answer, seconds=elapsed, passed=passed))

report = dict(model=MODEL_ID, revision=REVISION, gpu=torch.cuda.get_device_name(0),
              load_seconds=load_seconds, peak_vram_mib=torch.cuda.max_memory_reserved(0) / 2**20,
              cases=results)
output_dir = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
(output_dir / "reader_smoke.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
print(json.dumps(report, ensure_ascii=False, indent=2))